In [1]:

import ast
import operator as op
import pandas as pd
import numpy as np
from seuif97 import *
from scipy.interpolate import interp1d, LinearNDInterpolator # импортируем методы интерполяции
from math import sqrt, sin, log 

# Расчёт вспомогательных функций
def calc_Pw(T):
        # Saturation pressure at a given temperature
        return pd.Series([tx2p(T[i],0)/ 0.0980665 for i in T.index],T.index)
    
def calc_Hw(T):
        # Enthalpy of water
        return pd.Series([tx2h(T[i],0) for i in T.index],T.index)/4.186
def calc_Hs(T):
        # Enthalpy of steam at the saturation point
        return pd.Series([tx2h(T[i],1) for i in T.index],T.index)/4.186

def calc_T(P):
        # Steam temperature at the saturation point at a given pressure
        return pd.Series([px2t((P[i]+1)* 0.0980665,1) for i in P.index],P.index)

def calc_H(P,T):
        Hs=pd.Series([pt2h((P[i]+1)* 0.0980665,T[i])  for i in T.index],T.index)
        Hs=Hs/4.186
        Hs_=pd.Series([tx2h(T[i],1) for i in T.index],T.index)
        Hs_=Hs_/4.186
        Hs[Hs_>Hs]=Hs_[Hs_>Hs]
        return Hs

def clip(df,min_,max_):
    return df.clip(min_,max_)


def add_curve(Curves,Name,X,F):
        n=np.shape(X);
        if len(n)==1: # Интерполяция одномерных функций
            Curves.update({Name:interp1d(X,F,bounds_error=False, fill_value='extrapolate')})
        else:         # Интерполяция многомерных функций
            Curves.update({Name:LinearNDInterpolator(X, F,rescale=True)})
        return Curves


class ExpressionEvaluator:
    def __init__(self, df):
        self.df = df
        self.ops = {
            ast.Add: op.add,
            ast.Sub: op.sub,
            ast.Mult: op.mul,
            ast.Div: op.truediv,
            ast.Pow: op.pow,
            ast.USub: op.neg,
        }
        
        self.functions = {
            'fig':self.calc_curve,
            'clip':clip,
            'tw2p':calc_Pw,
            'tw2h':calc_Hw,
            'ts2h':calc_Hs,
            'px2t':calc_T,
            'pt2h':calc_H,
            'sqrt': np.sqrt,
            'sin': np.sin,
            'log': np.log,
            'sum': np.sum,
            # Добавьте другие функции по необходимости
        }
        self.curvs={}
    def calc_curve(self,Name,*X):
        #print('calc_curve')
        #print('Name:',Name)
        #print('X:',*X)
        return self.curvs[Name](*X)
        
    def add_curve(self,Name,X,F):
        n=np.shape(X);
        if len(n)==1: # Интерполяция одномерных функций
            self.curvs.update({Name:interp1d(X,F,bounds_error=False, fill_value='extrapolate')})
        else:         # Интерполяция многомерных функций
            self.curvs.update({Name:LinearNDInterpolator(X, F,rescale=True)})
        return self.curvs
    

    def eval_expr(self, expr):
        node = ast.parse(expr, mode='eval')
        return self._eval(node.body)

    def _eval(self, node):
        if isinstance(node, ast.Num):  # Число
            return node.n
        elif isinstance(node, ast.Str):  # Строковый литерал
            return node.s    
        elif isinstance(node, ast.Name):  # Столбец DataFrame
            return self.df[node.id]
        elif isinstance(node, ast.BinOp):  # Бинарная операция (+, -, *, /)
            left = self._eval(node.left)
            right = self._eval(node.right)
            return self.ops[type(node.op)](left, right)
        elif isinstance(node, ast.UnaryOp):  # Унарная операция (например, -x)
            return self.ops[type(node.op)](self._eval(node.operand))
        elif isinstance(node, ast.Call):  # Функции (sqrt(), sin() и т.д.)
            func_name = node.func.id
            args = [self._eval(arg) for arg in node.args]
            return self.functions[func_name](*args)
        else:
            raise ValueError(f"Неподдерживаемая операция: {type(node).__name__}")

    def calc_expr(self,expr,new_column_name):
        # Вычисляем выражение
        try:
            result = self.eval_expr(expr)
            print(f"Выражение:{new_column_name} = {expr} OK!\n") #Результат: {result}\n
        except Exception as e:
            print(f"Ошибка в выражении '{new_column_name}={expr}': {str(e)}")
            result = df.eval(expression)
        
        # Если результат - Series (один столбец), добавляем в DataFrame
        if isinstance(result, (pd.Series, np.ndarray)):
            self.df[new_column_name] = result
        else:
            # Если результат скалярный, применяем ко всем строкам
            self.df[new_column_name] = result
        return self.df    
        
    def calc_expressions(self,expressions):
        # Вычисление выражений
        for expr, col_name in expressions:
            print(col_name,'=',expr)
            self.calc_expr(expr, col_name)
        return  self.df 
        
    def calc_expressions_eq(self,expressions):
        # Вычисление выражений
        for expression in expressions:
            col_name, expr = expression.split('=')
            #print(col_name,'=',expr)
            self.calc_expr(expr, col_name)
        return  self.df 





C:\Users\NekliudovAV\AppData\Local\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
def Example():
        # Функция содержит пример кострукций    
        expressions_eq=['TsKPU=Tkpu',
                'TcKPU=Tr_KPU',
                'Dsp=D2_5',
                'GsuvEG=clip(GsuvEG,0,10000)',
                'GKPU=GsuvEG',
                'Tt=(Tsob1+Tsob2)/2',       # Усреднение температуры
                
                'H0=pt2h(P0,T0)',           # Enthalpy of superheated steam
                'Hsp=pt2h(Psp,Tsp)',        # Enthalpy of industrial extraction
                'Hst=pt2h(Pt,Tt)',          # Enthalpy of heat extraction
                'HcPSG=tw2h(TcPSG)',        # Enthalpy of PSG condensate
                'HsKPU=ts2h(TcKPU)',        # Enthalpy of steam at KPU
                'dT_c=TcPSG-Tr_PSG',        # PSG temperature undercooling
                'HwKPU=tw2h(TsuvEG)',       # Enthalpy of SUV before KPU
                'Hw_KPU=tw2h(Tr_KPU)',      # Enthalpy of SUV after KPU

                'HwPSG=tw2h(TrPSG)',        # Enthalpy of water before PSG
                'Hw_PSG=tw2h(Tr_PSG)',      # Enthalpy of water after PSG

                'Tt_c=tw2p(Pt)',            # Calculation of Pt_c condensation temperature in PSG by pressure Pt
                'PcPSG=tw2p(TcPSG)',     # Saturated steam pressure at PSG condensate temperature
                'Pt_plus_1=Pt+1',           # Pressure in absolute units kgf/cm2
                'P_PSG=tw2p(Tr_PSG)',     # Saturated steam pressure at temperature

                #"D0_=fig('D0',Pt,N)"
               ]

        # Загрузка исходных временных рядов
        # Data Reading
        FileName='TA5.xlsx'
        DF5_P=pd.read_excel(FileName)
        DF5_P=DF5_P.reset_index().drop(columns='index')
        DF5_P['time']=pd.to_datetime(DF5_P['time'])
        DF5_P=DF5_P.set_index('time')
        DF5_P.columns=[i[4:] for i in DF5_P.keys()]
        DF5_P.head()
        
        evaluator = ExpressionEvaluator(DF5_P)
        # Расчёт D0
        #D0=pd.read_excel('D0Curve.xlsx')
        #evaluator.add_curve('D0',D0[['Pt','N']],D0[['D0']])
        # Расчёт по уравнениям
        result = evaluator.calc_expressions_eq(expressions_eq) 
        return result 

In [5]:
Example()

Выражение:TsKPU = Tkpu OK!

Выражение:TcKPU = Tr_KPU OK!

Выражение:Dsp = D2_5 OK!

Выражение:GsuvEG = clip(GsuvEG,0,10000) OK!

Выражение:GKPU = GsuvEG OK!

Выражение:Tt = (Tsob1+Tsob2)/2 OK!

Выражение:H0 = pt2h(P0,T0) OK!

Выражение:Hsp = pt2h(Psp,Tsp) OK!

Выражение:Hst = pt2h(Pt,Tt) OK!

Выражение:HcPSG = tw2h(TcPSG) OK!

Выражение:HsKPU = ts2h(TcKPU) OK!

Выражение:dT_c = TcPSG-Tr_PSG OK!

Выражение:HwKPU = tw2h(TsuvEG) OK!

Выражение:Hw_KPU = tw2h(Tr_KPU) OK!

Выражение:HwPSG = tw2h(TrPSG) OK!

Выражение:Hw_PSG = tw2h(Tr_PSG) OK!

Выражение:Tt_c = tw2p(Pt) OK!

Выражение:PcPSG = tw2p(TcPSG) OK!

Выражение:Pt_plus_1 = Pt+1 OK!

Выражение:P_PSG = tw2p(Tr_PSG) OK!



,D0,D2_5,D7t,G_HOV,G_tsPSG,GmuwPSG,GrPSG,GsuvEG,Gsuv_PSG,H0,...,HsKPU,dT_c,HwKPU,Hw_KPU,HwPSG,Hw_PSG,Tt_c,PcPSG,Pt_plus_1,P_PSG
time,,,,,,,,,,,,,,,,,,,,,
2024-01-01 00:00:00,155.976190,36.599998,0.0,805.000000,0.0,0.0,805.000000,155.000000,805.000000,774.184274,...,613.514841,7.800003,21.047490,37.029029,18.047763,94.076963,0.006233,1.102396,1.000000,0.831497
2024-01-01 01:00:00,157.487179,39.000000,0.0,840.052632,0.0,0.0,840.052632,271.526316,840.052632,774.547091,...,613.108974,7.736842,21.047490,36.083177,18.047763,93.123891,0.006233,1.063653,1.000000,0.802744
2024-01-01 02:00:00,159.697674,40.074999,0.0,903.550000,0.0,0.0,903.550000,233.050000,903.550000,774.984445,...,612.185364,8.564999,21.047490,33.933897,18.047763,90.959229,-10.197162,1.014587,0.945000,0.740474
2024-01-01 03:00:00,160.454545,39.000000,0.0,830.000000,0.0,-0.5,830.500000,264.000000,830.000000,775.057246,...,612.657569,8.500000,21.047490,35.032203,18.047763,92.065217,0.006233,1.052765,1.000000,0.771771
2024-01-01 04:00:00,160.441860,38.340000,0.0,834.000000,0.0,0.0,834.000000,208.550000,834.000000,775.276423,...,612.228312,9.000000,21.047490,34.033745,18.047763,91.059760,0.006233,1.034176,1.000000,0.743275
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-02 09:00:00,0.000000,0.991111,0.0,293.000000,0.0,0.0,293.000000,0.000000,293.000000,678.429289,...,607.048077,5.997778,22.047072,22.047072,24.045818,25.045004,-10.197162,0.045847,0.997778,0.032322
2025-07-02 10:00:00,0.000000,1.000000,0.0,279.113636,0.0,0.0,279.113636,0.000000,279.113636,676.517483,...,607.048077,5.068182,22.183367,22.047072,24.045818,25.839731,0.006233,0.045498,1.000000,0.033887
2025-07-02 11:00:00,0.000000,1.000000,0.0,246.227273,0.0,0.0,246.227273,0.000000,246.227273,674.758885,...,607.048077,4.000000,23.046510,22.047072,24.749802,26.044079,0.006233,0.043304,1.000000,0.034300
